In [16]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# --- Toy Dataset (Same as the manual example) ---
# Features (Columns) represent word counts for:
# [Game, Score, Data, Algorithm, Win]
# Class 0 = Sports, Class 1 = Tech

X = np.array([
    [3, 2, 0, 0, 1], # Doc 1: Sports
    [2, 3, 0, 0, 0], # Doc 2: Sports
    [0, 0, 4, 1, 0], # Doc 3: Tech
    [0, 0, 5, 2, 0], # Doc 4: Tech
    [1, 1, 1, 1, 1]  # Doc 5: Sports (Mixed)
])
y = np.array([0, 0, 1, 1, 0])

# New Data to Classify: [Game: 1, Score: 0, Data: 1, Algorithm: 1, Win: 0]
X_new = np.array([
    [1, 0, 1, 1, 0]
])

# 1. Initialize the Multinomial Naive Bayes Model
# The 'alpha' parameter defaults to 1.0, implementing Laplace (Add-1) smoothing.
nb_model = MultinomialNB(alpha=1.0)

# 2. Train the model (Fit)
# In real scenarios, you would split the data into training and testing sets,
# but for this small example, we train on the entire set.
nb_model.fit(X, y)

# 3. Make predictions on the new sample
prediction = nb_model.predict(X_new)

# 4. Get probability estimates (A key output of NB)
# This returns the log probabilities P(C|X) for each class.
log_proba = nb_model.predict_log_proba(X_new)

# 5. Get actual probability estimates
proba = nb_model.predict_proba(X_new)


print("--- Scikit-learn Multinomial Naive Bayes Results ---")
print(f"Test Sample: {X_new[0]}")
print(f"Predicted Class (0=Sports, 1=Tech): {prediction[0]}")

print("\n--- Model Output Details ---")
print("Log Probability for each class (0, 1):")
# The class with the highest (least negative) log probability is the prediction
print(log_proba[0])

print("Actual Probability for each class (0, 1):")
# The class with the highest probability is the prediction
print(proba[0])

# --- Optional: Check Model's Learned Parameters ---
print("\n--- Learned Parameters ---")
# log_prior_: log(P(C)) - log of the prior probability for each class
print(f"Log Class Priors (log(P(C))): {nb_model.class_log_prior_}")
# feature_log_prob_: log(P(x_i|C)) - log of the likelihood for each feature given the class
print("Log Likelihoods for each Feature (log(P(x_i|C))):")
print(f"Class 0 (Sports): {nb_model.feature_log_prob_[0]}")
print(f"Class 1 (Tech):   {nb_model.feature_log_prob_[1]}")
print("--------------------------------------------------")

--- Scikit-learn Multinomial Naive Bayes Results ---
Test Sample: [1 0 1 1 0]
Predicted Class (0=Sports, 1=Tech): 1

--- Model Output Details ---
Log Probability for each class (0, 1):
[-1.02791694 -0.44277983]
Actual Probability for each class (0, 1):
[0.3577514 0.6422486]

--- Learned Parameters ---
Log Class Priors (log(P(C))): [-0.51082562 -0.91629073]
Log Likelihoods for each Feature (log(P(x_i|C))):
Class 0 (Sports): [-1.09861229 -1.09861229 -2.35137526 -2.35137526 -1.94591015]
Class 1 (Tech):   [-2.83321334 -2.83321334 -0.53062825 -1.44691898 -2.83321334]
--------------------------------------------------


Bernoulli NB is specifically designed for features that are binary (0 or 1), representing the presence or absence of an event (like a word existing in a document). Since our existing toy data is a count matrix, the most important step is to binarize it before fitting the BNB model.

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
# We import GaussianNB instead of MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import pandas as pd

In [4]:
# -------------------- Load the data set -------------------- 
data=pd.read_csv("SMSSpamCollection.csv",sep="\t")

In [13]:
data

,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
0,ham,Ok lar... Joking wif u oni...
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...
2,ham,U dun say so early hor... U c already then say...
3,ham,"Nah I don't think he goes to usf, he lives aro..."
4,spam,FreeMsg Hey there darling it's been 3 week's n...
...,...,...
5566,spam,This is the 2nd time we have tried 2 contact u...
5567,ham,Will ü b going to esplanade fr home?
5568,ham,"Pity, * was in mood for that. So...any other s..."
5569,ham,The guy did some bitching but I acted like i'd...


In [5]:
# -------------------- Check top 5 Values --------------------
print("First 5 Rows:")
print(data.head())

First 5 Rows:
    ham  \
0   ham   
1  spam   
2   ham   
3   ham   
4  spam   

  Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...  
0                      Ok lar... Joking wif u oni...                                                               
1  Free entry in 2 a wkly comp to win FA Cup fina...                                                               
2  U dun say so early hor... U c already then say...                                                               
3  Nah I don't think he goes to usf, he lives aro...                                                               
4  FreeMsg Hey there darling it's been 3 week's n...                                                               


In [6]:
# -------------------- Check the shape of data --------------------
print("\nShape of Dataset:")
print(data.shape)


Shape of Dataset:
(5571, 2)


In [7]:
# -------------------- Check the info --------------------
print("\nDataset Information:")
print(data.info())


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 2 columns):
 #   Column                                                                                                           Non-Null Count  Dtype
---  ------                                                                                                           --------------  -----
 0   ham                                                                                                              5571 non-null   str  
 1   Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...  5571 non-null   str  
dtypes: str(2)
memory usage: 542.8 KB
None


In [8]:
# -------------------- Check Missing Values --------------------
print("\nMissing Values:")
print(data.isnull().sum())


Missing Values:
ham                                                                                                                0
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...    0
dtype: int64


In [9]:
# -------------------- Check Duplicate Values --------------------

print("\nDuplicate Rows:")
print(data.duplicated().sum())


Duplicate Rows:
403


In [10]:
# -------------------- Remove Duplicate Values --------------------

df = data.drop_duplicates()

print("\nShape After Removing Duplicates:")
print(df.shape)


Shape After Removing Duplicates:
(5168, 2)


In [15]:
# -----------------------------------------------------
# 2. Convert Labels
# ham = 0
# spam = 1
# -----------------------------------------------------

df["ham"] = df["ham"].map({"ham":0, "spam":1})
df

,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
0,NaN,Ok lar... Joking wif u oni...
1,NaN,Free entry in 2 a wkly comp to win FA Cup fina...
2,NaN,U dun say so early hor... U c already then say...
3,NaN,"Nah I don't think he goes to usf, he lives aro..."
4,NaN,FreeMsg Hey there darling it's been 3 week's n...
...,...,...
5566,NaN,This is the 2nd time we have tried 2 contact u...
5567,NaN,Will ü b going to esplanade fr home?
5568,NaN,"Pity, * was in mood for that. So...any other s..."
5569,NaN,The guy did some bitching but I acted like i'd...
